In [85]:
#!pip install spectral-cube
#!pip install astroquery
#!pip install reproject
!pip install ccdproc
!pip install astropy pandas

   ---------------------------------------- 0.0/740.3 kB ? eta -:--:--
   ---------------------------------------- 740.3/740.3 kB 5.4 MB/s  0:00:00

   ---------------------------------------- 0/2 [astroscrappy]
   -------------------- ------------------- 1/2 [ccdproc]
   -------------------- ------------------- 1/2 [ccdproc]
   -------------------- ------------------- 1/2 [ccdproc]
   -------------------- ------------------- 1/2 [ccdproc]
   -------------------- ------------------- 1/2 [ccdproc]
   ---------------------------------------- 2/2 [ccdproc]



In [2]:
import glob #importing glob library
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np

import astropy
import astropy.units as u
from astropy.utils.data import download_file
from astropy.io import fits  # We use fits to open the actual data file
from astropy.utils import data
from astropy.wcs import wcs
from astropy.coordinates import SkyCoord
from astropy.time import Time
import pandas as pd
from astropy.nddata import CCDData

import ipywidgets as widgets
from IPython.display import display
import os
import ccdproc
import warnings

In [3]:
fits_files = glob.glob('C:/Users/jules/OneDrive/Desktop/Research Test Files/*.fit')

In [4]:
raw_fits_files = glob.glob('C:/Users/jules/OneDrive/Desktop/Test Flat/*.fit')
mflat = 'C:/Users/jules/OneDrive/Desktop/Test Flat/mflat.fits'

In [5]:
all_fits_files = glob.glob('D:/EB NSVS01031772 Usable Files/*.fit')

In [6]:
#hdul_test = fits.open('D:/EB NSVS01031772 Usable Files/2016 EB Files//03-30-2016/eb_R_90s_160329-059.fit')
#hdul_test[0].header

In [7]:
def file_by_date (directory):
    raw_fits_date = []
    fits_times = []

    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith(('.fits', '.fit')):
                file_path = os.path.join(root, filename)

                try:
                    with fits.open(file_path) as hdul:
                        date = hdul[0].header['DATE-OBS']
                        size = hdul[0].header['NAXIS1']

                    t = Time(date, format='isot', scale='utc')
                    MJD_fits = t.mjd
            
                    raw_fits_date.append({
                        'File Name': filename,
                        'Date': date,
                        'MJD': float(MJD_fits),
                        'Location': root,
                        'Size': size
                    })
            
                except (KeyError, OSError) as e: 
                    print(f"Skipping {filename}: {e}")

    raw_fits_date.sort(key=lambda x: x['Date'])

    return raw_fits_date

raw_fits_dict = file_by_date('C:/Users/jules/OneDrive/Desktop/Test Flat/')

In [8]:
#raw_fits_dict

In [9]:
def search_file(search_query, raw_fits_dict): #searchable dictionary for the flat for loop
    for i in raw_fits_dict:
        if i.get('File Name') == search_query:
            return os.path.join(i['Location'], i['File Name'])
        
    print(f"Error: Key '{search_query}' not found in the dictionary.")
    return None

search_file('test-0001.fit', raw_fits_dict)

'C:/Users/jules/OneDrive/Desktop/Test Flat/test-0001.fit'

In [10]:
warnings.filterwarnings('ignore', category=Warning, append=True)
master_flat = CCDData.read(mflat, unit='adu', ignore_missing_simple=True)
processed_data = []

for i in raw_fits_dict:
    if i['Size'] == 2048:
        raw_file_path = search_file(i['File Name'], raw_fits_dict)
        
        with fits.open(raw_file_path) as hdul:
            fits_reading = CCDData.read(raw_file_path, unit='adu', ignore_missing_simple=True)
            corrected_image = ccdproc.flat_correct(fits_reading, flat=master_flat)
            raw_data = hdul[0].data
            raw_header = hdul[0].header
            flat_data = corrected_image.data
            
            processed_data.append({
                'metadata': i,
                'image data': flat_data,
                'header': raw_header
            })

In [11]:
#flat_fits

In [12]:
def folder_by_date(processed_data):
    current_folder_date = ""
    prev_mjd = None

    for i in processed_data:
        item = i['metadata']
        mjd = item['MJD']
        flat_hdu = fits.PrimaryHDU(data=i['image data'], header=i['header'])
        
        if prev_mjd is None:
            current_folder_date = Time(mjd, format='mjd').iso.split()[0]
            
        else:
            if mjd - prev_mjd >= 0.104167:
                current_folder_date = Time(mjd, format='mjd').iso.split()[0]
                
        new_dir = f'C:/Users/jules/OneDrive/Desktop/EB_{current_folder_date}'
        os.makedirs(new_dir, exist_ok=True)
        
        file_output = os.path.join(new_dir, item['File Name'])
        flat_hdu.writeto(file_output, overwrite=True)  
        
        prev_mjd = mjd

In [14]:
folder_by_date(processed_data)

In [23]:
cube = np.zeros((len(raw_fits_files)+1,2048,2048))

In [24]:
def cubed_normed_fits (directory):
    norm_fits = []
    
    for root, dirs, file in os.walk(directory):
        for filename in file:
            if filename.endswith(('fit', 'fits')):
                full_path = os.path.join(root, filename)

                try:
                    with fits.open(full_path) as hdu_fits:
                        data_org = hdu_fits[0].data
                        data_max = np.max(data_org)
                        data_norm = data_org / data_max
                        norm_fits.append(data_norm)
                
                    for i, data_array in enumerate(norm_fits):
                        cube[i,:,:] = data_array

                except (KeyError, OSError) as e: 
                    print(f"Skipping {item['File Name']}: {e}")

    hdu_norm_new = fits.PrimaryHDU(cube)
    hdu_norm_new.writeto('cube_norm.fits', overwrite=True)

    cube_norm = fits.open('cube_norm.fits', memmap = False) 
    cube_norm_data = cube_norm[0].data

    return cube_norm_data

cube_norm_fits = cubed_normed_fits('C:/Users/jules/OneDrive/Desktop/Test Flat/')

In [25]:
print(cube_norm_fits)

[[[0.88829356 0.89523494 0.89568758 ... 0.89784706 0.89087033 0.87391531]
  [0.86770582 0.89568037 0.89023638 ... 0.88552862 0.8864947  0.86664903]
  [0.87716424 0.88365459 0.88704449 ... 0.88086969 0.87888467 0.86951029]
  ...
  [0.87031257 0.88993508 0.88975781 ... 0.89955759 0.89930248 0.87285763]
  [0.88468105 0.88635391 0.87544668 ... 0.89183736 0.88997358 0.86690807]
  [0.85787421 0.87863576 0.88715762 ... 0.88761669 0.88648129 0.85169119]]

 [[0.04296158 0.04289936 0.04426816 ... 0.04356821 0.04442371 0.0430238 ]
  [0.04229274 0.04408151 0.04237051 ... 0.04475035 0.04383263 0.04283714]
  [0.04324156 0.04433038 0.04352154 ... 0.04531031 0.04551252 0.04361487]
  ...
  [0.04272826 0.0443926  0.0447659  ... 0.04313268 0.04342822 0.04226163]
  [0.04331933 0.04373931 0.04316379 ... 0.04324156 0.04269715 0.04151501]
  [0.04241717 0.04363042 0.04381708 ... 0.04397262 0.04327267 0.04156167]]

 [[0.04269051 0.04494557 0.04401244 ... 0.04384137 0.04534992 0.04248834]
  [0.04214619 0.043157

In [34]:
file_select = widgets.Select(
    options=[item['File Name'] for item in raw_fits_dict],
    description='Display:',
    disabled=False
)

In [39]:
def norm_fits(selected_file):
    hdul_norm = fits.open(selected_file)
    data_org = hdul_norm[0].data
    data_max = np.max(data_org)
    data_norm = data_org / data_max
    hdul_norm.close()
    return data_norm

In [44]:
def fits_plot(data, filename):
    plt.figure(figsize=(24, 12))
    plt.imshow(data, cmap='gray', origin='lower')
    plt.colorbar(label='Intensity')
    plt.title(f'data from {raw_fits_dict[0]['File Name']}')
    plt.show()

In [45]:
def interactive_handler(selected_file):
    file_location = raw_fits_dict[0]['Location']
    full_path = os.path.join(file_location, selected_file)
    
    normalized_data = norm_fits(full_path)
    fits_plot(normalized_data, full_path)

widgets.interactive(interactive_handler, selected_file=file_select)

interactive(children=(Select(description='Display:', index=3, options=('mflat.fits', 'test-0001.fit', 'test-00…